**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**08 | Frozen Final Model Evaluation and Research Synthesis**

![Frozen primary Random Forest and secondary Dilated TCN: Essex one-time, participant-independent evaluation protocol](../../images/08_consolidated_workflow.svg)

**Final research question:** After seven source-governed development stages, how well do two **pre-frozen models** estimate the original unitless FE-model e1 and e3 strain curves on four previously unseen Test participants, and what do independent Test waveform, peak-value, peak-position and participant-level errors support?

**Pre-Test selection and source evidence:** The original Essex Figshare v13 source contains 1,897 trials, 1,729 valid IMU–FE pairs, 37 eligible participants, original IMU (101×42) and native FE (11×2) targets. Stages 04–07 established participant-disjoint provenance and model selection without using the four reserved Test subjects. The locked Stage 07 Training-only decision chose **Random Forest** as the *primary* model (CV subject-balanced waveform MAE 0.0007950). **Dilated TCN** is retained as one explicitly predeclared *secondary*, descriptive biomechanical peak comparator. Neither may replace the other on the basis of Test results.

**Frozen training recipe:** Before opening Test, fit both predeclared models *once* using all **33 development participants (Training + previously inspected Validation, 1,546 trials)**. This combines the established development splits **after** architecture, features, random seeds, training budget and metrics were frozen; it does not reuse Validation for new hyperparameter selection. The Random Forest uses the original five statistics per 42 IMU channels (210 predictors), with 200 trees, minimum leaf size 3 and seed 2026. Dilated TCN uses the 101×42 original signals and exactly eleven FE source indices, fixed 35 epochs, peak weight 0.25 and seed 2128. Its result is a fixed secondary diagnostic, not a second model-selection opportunity.

**One-time Test release:** A separate, committed [frozen final protocol](../../output/08_frozen_final_protocol.json) records the model identities, fitting policy and evaluation metrics **before** Test access. The notebook trains the fixed models first, then **stops by default** at the explicit release cell. Review the freeze and enter the exact documented approval phrase to authorise a **single final evaluation of four Test participants (183 trials)**. A local exclusive-release marker and detection of existing Test files block accidental reruns in the same working checkout. They cannot guarantee a global lock across unrelated computers.

**Scientific reporting:** The principal metric is **equal-participant-weighted waveform MAE**, averaging both FE targets and all original eleven source coordinates within each person. Report e1/e3 pointwise MAE/RMSE, peak-value errors, native-grid peak-index errors, trial-, activity- and participant-level heterogeneity, and exploratory participant-cluster bootstrap intervals (only four Test participants). No Test-based hyperparameter changes, threshold decisions, ranking changes or post-hoc candidate promotion are permitted. FE targets are unitless model-derived estimates, not measured physiological ground truth or independently validated clinical outcomes.

**Execution status:** This is an **unexecuted, pre-Test-frozen** research implementation. No Test score, clinical conclusion, successful Stage 08 Quality Gate or project-completion claim is made before actual authorised evaluation. After the executed notebook and its outputs are committed, final README and scientific synthesis documentation can be updated without another modelling notebook.


**00. Colab Repository Setup**

Use the existing shared Git synchroniser without changing HARTH or previously completed Essex notebooks. Open a fresh runtime or an appropriately clean checkout.


In [ ]:
# ======================== 00. Colab Repository Setup =========================

from pathlib import Path
import subprocess
import sys

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_branch = "main"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "--branch", repo_branch,
                    repo_url, str(repo_path)], check=True)
elif (
    not (repo_path / "python_modules/core/repository_sync.py").is_file()
    or "def sync_repository(" not in (
        repo_path / "python_modules/core/repository_sync.py"
    ).read_text(encoding="utf-8")
):
    cmd = ["git", "-C", str(repo_path)]
    origin = subprocess.check_output(cmd + ["remote", "get-url", "origin"],
                                     text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != repo_url.removesuffix(".git"):
        raise RuntimeError("Existing repository has an unexpected origin.")
    branch = subprocess.check_output(cmd + ["branch", "--show-current"],
                                     text=True).strip()
    dirty = subprocess.check_output(cmd + ["status", "--porcelain"],
                                    text=True).strip()
    if branch != repo_branch or dirty:
        raise RuntimeError("Cannot migrate a modified or different branch.")
    subprocess.run(cmd + ["pull", "--ff-only", "origin", repo_branch], check=True)
    if "python_modules.core.repository_sync" in sys.modules:
        raise RuntimeError(
            "Repository updated, but the old sync module is already loaded. "
            "Restart the kernel and rerun from the first cell."
        )

sys.path.insert(0, str(repo_path))
from python_modules.core.repository_sync import sync_repository

repo_path = sync_repository(repo_url, repo_branch, parent=repo_path.parent)
%cd {repo_path}


**01. Imports, Shared Device and Final Research Environment**

Retain the shared computer_device() helper from HARTH. The primary Random Forest runs on CPU through scikit-learn; the temporal secondary uses the detected PyTorch compute device. The device is reported, not manually selected.


In [ ]:
# ================== 01. Frozen Final Research Environment ===================
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

import hashlib
import json
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab
from core.computer_device import computer_device
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, ensure_pinned_matlab_archive,
    inspect_matlab_archive, extract_matlab_files,
)
from essex_numeric_recovery import check_record_registry
from essex_paired_dataset import (
    build_paired_arrays, paired_source_digest, save_ignored_cache,
)
from essex_timebase_audit import EXPECTED_CHANNELS
from essex_temporal_regression import check_temporal_arrays
from essex_regression_baselines import (
    regression_metrics, participant_metrics, trial_predictions,
    subject_balanced_mae,
)
from essex_stage08_final import (
    EXPECTED_PRIMARY, SECONDARY, PROTOCOL_FILE, FINAL_GATE_FILE,
    RELEASE_PHRASE, file_sha256, load_frozen_protocol,
    fit_frozen_primary, predict_frozen_primary,
    fit_frozen_secondary, predict_frozen_secondary,
    release_one_time_test, participant_bootstrap, paired_test_bootstrap,
)

plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100, figure_size=(9, 4))

study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
ledger_dir = processed_dir / "stage08_one_time_release"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# The common helper reports OS, RAM, GPU, VRAM and PyTorch CUDA readiness.
device = computer_device()
DEVICE = device.type
print("Frozen primary: Random Forest | secondary: Dilated TCN")
print("Scientific output directory:", output_dir)


**02. Verify Frozen Model Decision, Participant Roles and Provenance**

Load all earlier quality gates and the committed Stage 08 protocol. Verify that the original source hash, Training-only candidate, final development size and single-Test declaration agree before reading or fitting any Test observations.


In [ ]:
# ================== 02. Pinned Stage 04 Evidence and Splits ===================
# -------------------------- 1. Load Saved Evidence --------------------------
paths04 = {
    "gate": output_dir / "04_dataset_and_split_quality_gate.json",
    "manifest": output_dir / "04_local_dataset_build_manifest.json",
    "trials": output_dir / "04_trial_split_assignments.csv",
    "subjects": output_dir / "04_subject_split_assignments.csv",
    "source": output_dir / "01_consolidated_source_and_trial_quality_gate.json",
    "registry": output_dir / "01_original_record_registry.csv",
    "mapping": output_dir / "03_original_time_coordinate_mapping.csv",
}
missing04 = [key for key, path in paths04.items() if not path.is_file()]
if missing04:
    raise FileNotFoundError("Missing committed upstream evidence: " +
                            ", ".join(missing04))
gate04 = json.loads(paths04["gate"].read_text(encoding="utf-8"))
manifest04 = json.loads(paths04["manifest"].read_text(encoding="utf-8"))
source01 = json.loads(paths04["source"].read_text(encoding="utf-8"))
assigned = pd.read_csv(paths04["trials"], dtype=str)
subjects04 = pd.read_csv(paths04["subjects"], dtype=str)
assert gate04["stage04_quality_gate"] == "PASSED"
assert gate04["source_article_id"] == ARTICLE_ID == 30225739
assert gate04["source_version"] == ARTICLE_VERSION == 13
assert gate04["original_sha256"] == source01["archive_sha256"]
assert gate04["original_sha256"] == manifest04["original_archive_sha256"]
assert gate04["canonical_pair_sha256"] == manifest04["canonical_pair_sha256"]
assert manifest04["feature_shape"] == [1729, 101, 42]
assert manifest04["target_shape"] == [1729, 11, 2]
assert gate04["original_source_empty_fe"] == 168
assert gate04["original_target_units_as_published"] == "unitless"

# ------------------------ 2. Freeze Participant Roles -----------------------
expected_splits = {"TRAIN": 1354, "VALIDATION": 192, "TEST": 183}
assert assigned["split"].value_counts().to_dict() == expected_splits
assert assigned["local_record_key"].is_unique
assert len(assigned) == 1729
assert subjects04["subject_id"].is_unique
assert subjects04["split"].value_counts().to_dict() == {
    "TRAIN": 29, "VALIDATION": 4, "TEST": 4,
    "EXCLUDED_NO_FE_TARGET": 3,
}
assert assigned.groupby("subject_id")["split"].nunique().eq(1).all()
assert assigned["split"].eq(
    assigned["subject_id"].map(subjects04.set_index("subject_id")["split"])
).all()
train_mask = assigned["split"].eq("TRAIN").to_numpy()
valid_mask = assigned["split"].eq("VALIDATION").to_numpy()
test_mask = assigned["split"].eq("TEST").to_numpy()
assert not np.any(train_mask & valid_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(valid_mask & test_mask)
print("Frozen Training trials:", train_mask.sum())
print("Frozen Validation trials:", valid_mask.sum())
print("Reserved Test trials (no model access):", test_mask.sum())



# ------------------------- 3. Check Final Freeze -----------------------------
required = {
    "stage05": output_dir / "05_regression_protocol_and_quality_gate.json",
    "stage06": output_dir / "06_temporal_protocol_and_quality_gate.json",
    "stage07": output_dir / "07_robustness_protocol_and_quality_gate.json",
    "decision": output_dir / "07_training_candidate_decision.csv",
    "freeze": output_dir / PROTOCOL_FILE,
}
for key, path in required.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing prerequisite {key}: {path}")
gate05 = json.loads(required["stage05"].read_text(encoding="utf-8"))
gate06 = json.loads(required["stage06"].read_text(encoding="utf-8"))
gate07 = json.loads(required["stage07"].read_text(encoding="utf-8"))
decision = pd.read_csv(required["decision"])
primary_row = decision.loc[decision["training_cv_selected"].eq(True)]
assert len(primary_row) == 1
assert gate06["quality_gate"] == "PASSED"
assert gate06["canonical_pair_sha256"] == gate04["canonical_pair_sha256"]
protocol, protocol_sha256 = load_frozen_protocol(
    required["freeze"], gate04, gate05, gate07,
    primary_row.iloc[0],
)
assert protocol["source_article_id"] == ARTICLE_ID == 30225739
assert protocol["source_version"] == ARTICLE_VERSION == 13
assert assigned["split"].value_counts().to_dict() == {
    "TRAIN": 1354, "VALIDATION": 192, "TEST": 183,
}
dev_mask = train_mask | valid_mask
assert dev_mask.sum() == 1546 and test_mask.sum() == 183
dev_info = assigned.loc[dev_mask].reset_index(drop=True)
test_info = assigned.loc[test_mask].reset_index(drop=True)
assert dev_info["subject_id"].nunique() == 33
assert test_info["subject_id"].nunique() == 4
assert set(dev_info["subject_id"]).isdisjoint(set(test_info["subject_id"]))
assert test_info["local_record_key"].is_unique
print("Pre-Test model selection: LOCKED", protocol["locked_primary_candidate"])
print("Frozen protocol SHA256:", protocol_sha256)
print("Development: 33 participants / 1546 trials")
print("Reserved Test: 4 participants / 183 trials (unopened)")


**03. Reconstruct Source-Exact Cache and Materialise Development Only**

The source archive and paired-data cache are checked with original SHA256 lineage, but only the 33-participant Training+Validation development arrays are copied for fitting. Neither the Test IMU trials nor the Test FE labels are materialised for model fitting at this stage.


In [ ]:
# ============== 03. Verified Cache and Source Reconstruction ================
# ------------------- 1. Pinned Original MATLAB Reader -----------------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat
assert version("mat-io") == "1.0.1"

# --------------------- 2. Rebuild Cache Only If Missing ----------------------
cache_path = processed_dir / "essex_imu_fe_pairs_v13.npz"
if not cache_path.exists():
    archive, hashes = ensure_pinned_matlab_archive(raw_dir)
    assert hashes["sha256"] == source01["archive_sha256"]
    inventory = inspect_matlab_archive(archive, expected_mat_files=40)
    mat_files = extract_matlab_files(archive, mat_dir, inventory)
    assert len(mat_files) == 40
    registry = pd.read_csv(paths04["registry"], dtype={
        "local_record_key": str, "source_file": str,
        "subject_id": str, "task": str, "updated_trial_id": str,
        "fe_representation": str,
    })
    mapping = pd.read_csv(paths04["mapping"], dtype={
        "local_record_key": str, "imu_position_indices": str,
        "mapping_status": str,
    })
    check_record_registry(registry, source01)
    rebuilt, eligible, excluded = build_paired_arrays(
        load_from_mat, mat_dir, registry, mapping, source01,
    )
    assert len(eligible) == 1729 and len(excluded) == 168
    assert eligible["local_record_key"].tolist() == (
        assigned["local_record_key"].tolist()
    )
    assert paired_source_digest(rebuilt, assigned) == (
        gate04["canonical_pair_sha256"]
    )
    save_ignored_cache(cache_path, rebuilt, assigned)
    del rebuilt, eligible, excluded
    print("Source-verified local cache reconstructed.")
else:
    print("Using existing Stage 04 cache, subject to fingerprint verification.")

# ---------------------- 3. Verify Cached Source Identity --------------------
with np.load(cache_path, allow_pickle=False) as saved:
    assert set(saved.files) == {
        "X_full", "X_at_fe", "y_fe", "imu_positions", "record_keys"
    }
    assert saved["record_keys"].astype(str).tolist() == (
        assigned["local_record_key"].tolist()
    )
    # Original Test source values are included ONLY in this integrity digest;
    # no Test feature/target is supplied to any estimator or metric below.
    source_digest = paired_source_digest(
        {key: saved[key] for key in (
            "X_full", "X_at_fe", "y_fe", "imu_positions"
        )}, assigned,
    )
    assert source_digest == gate04["canonical_pair_sha256"]
    x_development = saved["X_full"][dev_mask].copy()
    idx_development = saved["imu_positions"][dev_mask].copy()
    y_development = saved["y_fe"][dev_mask].copy()
assert x_development.shape == (1546, 101, 42)
assert idx_development.shape == (1546, 11)
assert y_development.shape == (1546, 11, 2)
check_temporal_arrays(x_development, idx_development, y_development)
print("Canonical source fingerprint: VERIFIED")
print("Only Development observations materialised before Test release.")


**04. Freeze Data Feature Contracts and Evaluation Rules**

Before fitting, assert native source coordinate alignment and the 210-feature Random Forest contract. There is no hyperparameter tuning, further validation search or additional model selection. The original Test partition remains sealed.


In [ ]:
# =================== 04. Final Development Protocol Checks ===================
from essex_regression_baselines import trial_features, flatten_targets

development_subjects = set(dev_info["subject_id"])
reserved_subjects = set(test_info["subject_id"])
assert len(development_subjects) == 33
assert len(reserved_subjects) == 4
assert not development_subjects & reserved_subjects
assert len(set(dev_info["local_record_key"])) == len(dev_info)
assert not set(dev_info["local_record_key"]) & set(
    test_info["local_record_key"]
)
features_dev, original_feature_names = trial_features(
    x_development, EXPECTED_CHANNELS
)
targets_dev = flatten_targets(y_development)
assert features_dev.shape == (1546, 210)
assert targets_dev.shape == (1546, 22)
assert protocol["final_fitting_policy"]["no_new_tuning_or_model_selection"]
assert protocol["primary_model"]["random_state"] == 2026
assert protocol["secondary_model"]["seed"] == 2128
assert protocol["secondary_model"]["epochs"] == 35
assert protocol["single_final_evaluation"]["evaluation_rounds"] == 1
print("Frozen development and model contracts: PASSED")
print("TEST data not materialised; no model selection remains.")


**05. Fit Both Frozen Models Exactly Once on Development**

Run the two predeclared fitting recipes on all 33 Development participants without accessing Test features or labels. The Random Forest is primary; the Dilated TCN is a fixed secondary peak-tradeoff comparator. Keep final fitting losses and model metadata in memory until the one-time release, so an unapproved run does not leave incomplete public Test artifacts.


In [ ]:
# =============== 05. Train Frozen Development Models Once ===================
primary, primary_features = fit_frozen_primary(
    x_development, y_development, EXPECTED_CHANNELS, protocol
)
secondary, secondary_scaler, final_training_history = fit_frozen_secondary(
    x_development, y_development, idx_development,
    protocol, device=device,
)
assert len(primary.estimators_) == 200
assert primary.n_outputs_ == 22
assert len(final_training_history) == 35
assert np.isfinite(final_training_history["training_loss"]).all()
secondary_parameters = sum(p.numel() for p in secondary.parameters())
model_manifest = {
    "stage": "08",
    "frozen_protocol_sha256": protocol_sha256,
    "model_selection_dataset": "TRAIN_OOF_ONLY",
    "development_subjects": 33,
    "development_trials": 1546,
    "original_imu_shape": [101, 42],
    "original_fe_shape": [11, 2],
    "primary": {
        "name": EXPECTED_PRIMARY,
        "trees": len(primary.estimators_),
        "features": len(primary_features),
        "seed": 2026,
    },
    "secondary": {
        "name": SECONDARY,
        "trainable_parameters": secondary_parameters,
        "epochs": len(final_training_history),
        "seed": 2128,
        "role": "secondary_only_no_test_promotion",
    },
    "selected_pytorch_device": DEVICE,
    "scikit_learn_version": version("scikit-learn"),
    "torch_version": torch.__version__,
}
print("FROZEN DEVELOPMENT FIT COMPLETE")
print("RF estimators:", len(primary.estimators_))
print("TCN parameters:", secondary_parameters)
print("The reserved Test set is still sealed.")


**06. Explicit One-Time Test Release — Human Approval Required**

**Deliberate scientific safeguard.** This cell is disabled by default and therefore stops Run All. Review the committed protocol, inspect the development-fit evidence and confirm that no previous final Test evaluation has been performed. Then replace the empty string with the exact phrase printed in this cell and run from here onward **once**. Do not reset or bypass the release ledger to search for a better Test score.


In [ ]:
# ================== 06. Explicit One-Time Test Release ======================
# Read the already committed frozen protocol before granting release.
print("Primary model:", protocol["locked_primary_candidate"])
print("Frozen protocol SHA256:", protocol_sha256)
print("Development fits completed:", len(primary.estimators_) == 200
      and len(final_training_history) == 35)
print("To authorise one-time Test evaluation, set:")
print('FINAL_TEST_APPROVAL = "I_APPROVE_ONE_TIME_TEST_EVALUATION"')

# STOP BY DEFAULT. This line is intentionally an empty string.
FINAL_TEST_APPROVAL = ""

release_receipt = release_one_time_test(
    approval=FINAL_TEST_APPROVAL,
    protocol=protocol,
    protocol_sha256=protocol_sha256,
    output_dir=output_dir,
    ledger_dir=ledger_dir,
)
assert release_receipt["status"] == "TEST_RELEASE_CONSUMED"
print("ONE-TIME FINAL TEST RELEASE AUTHORISED")


**07. Single Predeclared Test Inference and Original-Grid Predictions**

After the exclusive release marker has been written, read the four reserved Test participants exactly once from the source-verified cache. Predict using the already fitted and frozen models, without fitting, selecting or tuning anything on these participants. Long-form outputs preserve original trial keys, 11 FE positions and e1/e3 targets.


In [ ]:
# ================= 07. Exactly One Reserved Test Inference ===================
assert release_receipt["protocol_sha256"] == protocol_sha256
assert release_receipt["status"] == "TEST_RELEASE_CONSUMED"

# ----------- First materialisation of reserved Test observations ------------
with np.load(cache_path, allow_pickle=False) as saved:
    x_test = saved["X_full"][test_mask].copy()
    pos_test = saved["imu_positions"][test_mask].copy()
    y_test = saved["y_fe"][test_mask].copy()
    assert saved["record_keys"].astype(str)[test_mask].tolist() == (
        test_info["local_record_key"].tolist()
    )
assert x_test.shape == (183, 101, 42)
assert pos_test.shape == (183, 11)
assert y_test.shape == (183, 11, 2)
check_temporal_arrays(x_test, pos_test, y_test)
assert test_info["subject_id"].nunique() == 4
assert not set(test_info["subject_id"]) & development_subjects

# ------------------- Readout from both locked estimators ---------------------
prediction_primary = predict_frozen_primary(
    primary, x_test, EXPECTED_CHANNELS, primary_features
)
prediction_secondary = predict_frozen_secondary(
    secondary, secondary_scaler,
    x_test, pos_test, device=device,
)
test_predictions = {
    EXPECTED_PRIMARY: prediction_primary,
    SECONDARY: prediction_secondary,
}
long_table = pd.concat([
    trial_predictions(
        y_test, prediction, test_info["local_record_key"],
        test_info["subject_id"], test_info["task"], model_name
    )
    for model_name, prediction in test_predictions.items()
], ignore_index=True)
assert len(long_table) == 183 * 11 * 2 * 2
assert set(long_table["subject_id"]) == reserved_subjects
assert long_table["local_record_key"].nunique() == 183
assert long_table["fe_position"].between(0, 10).all()
assert np.isfinite(long_table["predicted"]).all()
long_table.to_csv(
    output_dir / "08_test_trial_predictions.csv", index=False
)
print("FROZEN TEST INFERENCE COMPLETE; no further Test fit permitted.")
print("Test trial/position/target/model rows:", len(long_table))


**08. Independent Test Metrics, Participant-Level Uncertainty and Activity Errors**

Report the predeclared primary subject-balanced MAE and e1/e3 waveform, peak-value and peak-index errors. The fixed secondary model can be compared descriptively but must never replace the primary model based on these Test findings. Bootstrap resamples whole Test participants (n=4), not individual timepoints or repeated trials; intervals are exploratory.


In [ ]:
# ================= 08. Predeclared Held-Out Test Evaluation ===================
rows, subject_frames, person_rows, bootstrap_rows = [], [], [], []
for model_name, prediction in test_predictions.items():
    balanced = subject_balanced_mae(
        y_test, prediction, test_info["subject_id"]
    )
    for m in regression_metrics(y_test, prediction):
        rows.append({
            "model": model_name,
            "role": "PRIMARY" if model_name == EXPECTED_PRIMARY
                    else "SECONDARY_DESCRIPTIVE",
            "subject_balanced_mae": balanced,
            "test_participants": 4, **m,
        })
    detail = participant_metrics(
        y_test, prediction,
        test_info["subject_id"], test_info["task"],
        model=model_name, phase="FINAL_TEST",
    )
    subject_frames.append(detail)
    for participant in sorted(reserved_subjects):
        mask = test_info["subject_id"].eq(participant).to_numpy()
        for met in regression_metrics(y_test[mask], prediction[mask]):
            person_rows.append({
                "model": model_name,
                "subject_id": participant, **met,
            })
    bootstrap_rows.append({
        "model": model_name,
        **participant_bootstrap(
            y_test, prediction, test_info["subject_id"],
            draws=2000, seed=2026,
        ),
    })
metrics_final = pd.DataFrame(rows)
subject_activity = pd.concat(subject_frames, ignore_index=True)
subject_final = pd.DataFrame(person_rows)
boot_final = pd.DataFrame(bootstrap_rows)
paired = paired_test_bootstrap(
    y_test, prediction_primary, prediction_secondary,
    test_info["subject_id"], draws=2000, seed=2026,
)
boot_final = pd.concat([
    boot_final,
    pd.DataFrame([{
        "model": "Dilated TCN minus frozen RF",
        "estimate": paired["difference"],
        "bootstrap_low_95": paired["bootstrap_low_95"],
        "bootstrap_high_95": paired["bootstrap_high_95"],
        "participants": paired["participants"],
        "resamples": paired["resamples"],
        "scope": paired["scope"],
    }]),
], ignore_index=True)
task_final = subject_activity.groupby(
    ["model", "task", "target"], as_index=False
).agg(
    subject_mean_mae=("mae", "mean"),
    subject_mean_peak_value_mae=("peak_value_mae", "mean"),
    subject_mean_peak_index_mae=("peak_index_mae", "mean"),
    test_subjects=("subject_id", "nunique"),
    trials=("trials", "sum"),
)
assert len(metrics_final) == 4
assert len(subject_final) == 2 * 4 * 2
assert len(boot_final) == 3
assert metrics_final["test_participants"].eq(4).all()

metrics_final.to_csv(
    output_dir / "08_test_primary_secondary_metrics.csv", index=False,
)
subject_activity.to_csv(
    output_dir / "08_test_subject_activity_metrics.csv", index=False,
)
subject_final.to_csv(
    output_dir / "08_test_subject_metrics.csv", index=False,
)
task_final.to_csv(
    output_dir / "08_test_task_metrics.csv", index=False,
)
boot_final.to_csv(
    output_dir / "08_test_subject_bootstrap.csv", index=False,
)
print("Final primary model remains:", EXPECTED_PRIMARY)
print("Test metrics are descriptive and must not change model selection.")
display(metrics_final)
display(boot_final)


**09. Final Scientific Figures and Source-Identified Waveforms**

Prepare grouped-bar figures for primary versus secondary Test waveform and peak-magnitude errors, participant-level heterogeneity and actual-versus-predicted FE curves for the lexicographically first Test trial. Figures use the original eleven FE indices, not interpolated time, independent real-world strain or clinical outcomes.


In [ ]:
# ======================= 09. Final Research Figures =========================
display_names = [EXPECTED_PRIMARY, SECONDARY]
chart = metrics_final.pivot(index="model", columns="target", values="mae")
chart = chart.reindex(display_names)
fig, ax = plt.subplots(figsize=(10, 4))
chart.plot.bar(ax=ax, width=0.75)
ax.set_title("One-time held-out Test waveform MAE")
ax.set_xlabel("Pre-frozen model: primary versus descriptive secondary")
ax.set_ylabel("MAE (unitless FE-model target)")
ax.tick_params(axis="x", rotation=12)
save_figure(fig, output_dir / "08_test_waveform_mae_comparison.png")
plt.show()

peak = metrics_final.pivot(
    index="model", columns="target", values="peak_value_mae"
).reindex(display_names)
fig, ax = plt.subplots(figsize=(10, 4))
peak.plot.bar(ax=ax, width=0.75)
ax.set_title("One-time held-out Test peak-value MAE")
ax.set_xlabel("No Test-driven candidate promotion")
ax.set_ylabel("Peak magnitude MAE (unitless FE-model target)")
ax.tick_params(axis="x", rotation=12)
save_figure(fig, output_dir / "08_test_peak_error_comparison.png")
plt.show()

per_person = subject_final.groupby(
    ["subject_id", "model"], as_index=False
)["mae"].mean().pivot(
    index="subject_id", columns="model", values="mae"
).reindex(columns=display_names)
fig, ax = plt.subplots(figsize=(10, 4))
per_person.plot.bar(ax=ax, width=0.75)
ax.set_title("Participant-level Test waveform errors (only four people)")
ax.set_xlabel("Frozen Test participant")
ax.set_ylabel("Mean e1/e3 MAE (unitless FE-model target)")
ax.tick_params(axis="x", rotation=0)
save_figure(fig, output_dir / "08_test_subject_errors.png")
plt.show()

example_i = int(np.argsort(
    test_info["local_record_key"].to_numpy()
)[0])
example_key = str(test_info.iloc[example_i]["local_record_key"])
assert example_key == sorted(test_info["local_record_key"])[0]
for j, target in enumerate(("e1", "e3")):
    fig, ax = plt.subplots(figsize=(9, 4))
    ticks = np.arange(11)
    ax.plot(ticks, y_test[example_i, :, j],
            marker="o", label="Original FE source")
    ax.plot(ticks, prediction_primary[example_i, :, j],
            marker="s", label="Frozen primary Random Forest")
    ax.plot(ticks, prediction_secondary[example_i, :, j],
            marker="^", label="Predeclared secondary TCN")
    ax.set_title(f"Fixed Test example: {target} | {example_key}")
    ax.set_xlabel("Original FE position (index 0–10)")
    ax.set_ylabel("Published unitless FE-model target")
    ax.set_xticks(ticks)
    ax.legend()
    save_figure(
        fig, output_dir / f"08_test_waveform_example_{target}.png"
    )
    plt.show()
print("Figures are final descriptive results, never a tuning criterion.")


**10. Final Quality Gate, Release Audit and Public Evidence Export**

A PASSED gate is recorded only after source integrity, pre-Test frozen protocol, explicit release receipt, exactly one inferential pass per predeclared model, participant isolation, full Test predictions and 16 required public outputs are verified. The model choice is not revised. Package the public `08_*` outputs for manual GitHub commit after this run.


In [ ]:
# ====================== 10. Stage 08 Final Quality Gate ======================
expected08 = [
    "08_frozen_final_protocol.json",
    "08_development_fit_manifest.json",
    "08_development_fit_loss_history.csv",
    "08_test_trial_predictions.csv",
    "08_test_primary_secondary_metrics.csv",
    "08_test_subject_activity_metrics.csv",
    "08_test_subject_metrics.csv",
    "08_test_task_metrics.csv",
    "08_test_subject_bootstrap.csv",
    "08_test_waveform_mae_comparison.png",
    "08_test_peak_error_comparison.png",
    "08_test_subject_errors.png",
    "08_test_waveform_example_e1.png",
    "08_test_waveform_example_e3.png",
    "08_final_test_release_receipt.json",
    "08_final_test_quality_gate.json",
]
assert protocol["protocol_status"] == "FROZEN_BEFORE_TEST"
assert file_sha256(required["freeze"]) == protocol_sha256
assert release_receipt["protocol_sha256"] == protocol_sha256
assert release_receipt["status"] == "TEST_RELEASE_CONSUMED"
assert gate07["quality_gate"] == "PASSED"
assert gate07["provisional_final_candidate_from_training_only"] == EXPECTED_PRIMARY
assert 33 == len(development_subjects)
assert 4 == len(reserved_subjects)
assert not development_subjects & reserved_subjects
assert len(long_table) == 2 * 183 * 22
assert set(metrics_final["model"]) == {EXPECTED_PRIMARY, SECONDARY}
assert set(metrics_final["role"]) == {"PRIMARY", "SECONDARY_DESCRIPTIVE"}
assert len(metrics_final) == 4
assert len(subject_final) == 16
assert boot_final["participants"].eq(4).all()
assert np.isfinite(long_table[["observed", "predicted"]]).all().all()
assert metrics_final[["mae", "rmse", "peak_value_mae"]].ge(0).all().all()
assert len(final_training_history) == 35

fit_file = output_dir / "08_development_fit_manifest.json"
fit_file.write_text(
    json.dumps(model_manifest, indent=2) + "\n", encoding="utf-8"
)
final_training_history.to_csv(
    output_dir / "08_development_fit_loss_history.csv", index=False
)
receipt_file = output_dir / "08_final_test_release_receipt.json"
receipt_file.write_text(
    json.dumps(release_receipt, indent=2) + "\n",
    encoding="utf-8",
)
missing = [name for name in expected08
           if name != FINAL_GATE_FILE and
           not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Incomplete Stage 08 evidence: " + ", ".join(missing)
    )

evidence_hashes = {
    name: file_sha256(output_dir / name)
    for name in expected08 if name != FINAL_GATE_FILE
}
final_record = {
    "stage": "08",
    "quality_gate": "PASSED",
    "research_status": "one_time_final_test_complete",
    "frozen_protocol_sha256": protocol_sha256,
    "source_archive_sha256": gate04["original_sha256"],
    "canonical_pair_sha256": source_digest,
    "primary_selected_pre_test": EXPECTED_PRIMARY,
    "primary_selection_stage": "07_training_only",
    "secondary_fixed_descriptive": SECONDARY,
    "post_test_model_selection": False,
    "final_fit_development_subjects": 33,
    "final_fit_development_trials": 1546,
    "test_subjects": 4,
    "test_trials": 183,
    "test_evaluations_authorised": 1,
    "test_inferences_per_predeclared_model": 1,
    "test_model_fitting": False,
    "test_model_tuning": False,
    "primary_subject_balanced_mae": float(
        metrics_final.loc[
            metrics_final["model"].eq(EXPECTED_PRIMARY),
            "subject_balanced_mae"
        ].iloc[0]
    ),
    "secondary_subject_balanced_mae": float(
        metrics_final.loc[
            metrics_final["model"].eq(SECONDARY),
            "subject_balanced_mae"
        ].iloc[0]
    ),
    "published_fe_target_unit": "unitless",
    "original_fe_positions": 11,
    "source_interpolation": False,
    "bootstrap_clusters": 4,
    "bootstrap_intervals_exploratory": True,
    "clinical_validation": False,
    "independent_external_dataset": False,
    "fitted_weights_committed": False,
    "selected_compute_device": DEVICE,
    "sklearn_version": version("scikit-learn"),
    "torch_version": torch.__version__,
    "public_evidence_sha256": evidence_hashes,
}
final_gate_path = output_dir / FINAL_GATE_FILE
if final_gate_path.exists():
    raise FileExistsError("Final Test gate exists; no duplicate scoring.")
final_gate_path.write_text(
    json.dumps(final_record, indent=2) + "\n",
    encoding="utf-8",
)
assert all((output_dir / name).is_file() for name in expected08)
print("STAGE 08 | FROZEN ONE-TIME TEST QUALITY GATE: PASSED")
print("Primary choice preserved:", EXPECTED_PRIMARY)
print("Final public evidence files:", len(expected08))
print("One-time Test participants:", len(reserved_subjects))
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="08",
)


**11. Final Scientific Interpretation and Portfolio Handover**

**Complete only from the executed final Test outputs.** Record the primary Random Forest and predeclared descriptive Dilated TCN participant-balanced Test MAE, target-specific waveform MAE/RMSE, peak-value error and peak-position error, and heterogeneity across the four Test participants and the six activities. Contrast Training development signals with the genuinely reserved same-source Test results; do not assume the Test ranking or errors in advance.

**Interpretation constraints:** The primary Random Forest was locked before Test; no post-Test re-selection is allowed, even if the secondary TCN has an advantage for a selected peak metric. The final fit used Training plus Validation *after* model selection, so previously reported Validation scores from earlier, differently fitted estimators are not directly interchangeable with the final Test scores. The four-person participant-cluster bootstrap is exploratory, not an independent external validation or a robust population-level confidence statement. The source FE values remain **unitless, simulation-derived estimates**, not observed bone strains or clinical evidence.

**Evidence traceability:** A release marker and a committed freeze protect the protocol locally and document explicit single-use Test authorisation. They do not constitute global enforcement across computers. The public output folder is intended to contain **16 Stage 08 CSV/JSON/PNG artifacts**, including the before-Test protocol and final gate. Trained models, private numerical caches and source archives are not published by this notebook.

**Project closure:** Once the final Test has genuinely run, its outputs have been committed and reviewed, prepare the professional [project README](../../README.md) and supporting methodology/results documentation: original Essex source, 01–08 workflow, original split design, training-only model choice, final once-only Test results, quality gates, limitations, reproducibility steps and clean portfolio presentation. Do not publish a fabricated completed-study claim before that last audit.
